# 03 · MCDA baseline with clinician weights

A transparent weighted score over five criteria, with weights proposed by NEBSL clinicians, compared with random order and a grade-only rule. Code: `matching/mcda.py`, `evaluation/metrics.py`.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

## 1. Criteria and weights

In [ ]:
from recommendation.matching.mcda import Weights, score

m = cfg["matching"]
w = Weights(**m["weights"])
print("weights version:", m["weights_version"])
pd.Series(w.normalised()).round(3).rename("weight").to_frame()

## 2. Score every candidate and rank within each request

In [ ]:
pairs = pd.read_excel(DATASET, sheet_name="Ranking_Pairs")
scored = score(pairs, w)
crit = [c for c in scored.columns if c.startswith("s_")]
scored.groupby("Label_Allocated")[crit + ["MCDA_Score"]].mean().T.rename(
    columns={0: "not chosen", 1: "chosen"}).round(3)

## 3. Results: all requests and held-out test requests

In [ ]:
from recommendation.evaluation.metrics import compare
from recommendation.evaluation.split import grouped_split

sp = cfg["split"]
_, test = grouped_split(scored, group_col=sp["group_col"], test_size=sp["test_size"], seed=sp["seed"])
display(compare(scored).round(3))
compare(test).round(3)

**Reading the table:** Top-3 = share of requests where the cornea NEBSL actually chose is in the top 3. These measure agreement with past decisions, not transplant outcomes.